# 11. Statistical Model Comparison: Bayesian ROPE, Friedman Ranks, and Corrected Resampling

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Dietterich (1998 *Approximate statistical tests for comparing supervised classification algorithms*); Nadeau & Bengio (2003 *Inference for the Generalization Error*); Demšar (2006 *Statistical Comparisons of Classifiers over Multiple Data Sets*); Benavoli et al. (2017 *Time for a Change: a Tutorial for Comparing 3+ Machine Learning Algorithms Through Bayesian Tests*).
- **Purpose:** Implement rigorous statistical comparisons for model benchmarking: McNemar's test on discordant pairs, Nadeau-Bengio corrected resampled t-test for cross-validation, non-parametric Friedman ranking, and Bayesian correlated t-tests with Region of Practical Equivalence (ROPE).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [10. AI Evaluation Science Bradley Terry IRT Dawid Skene and Min-K%](10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.ipynb) $\to$ `🔗 Downstream Syntheses:` [12. Topological Data Analysis Persistent Homology and Singular Learning LLC](12_topological_data_analysis_persistent_homology_and_singular_learning_llc.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never use an uncorrected paired t-test on cross-validation splits (overlapping training data induces strong positive correlation, causing up to 50% false positive rates); do not average raw benchmark accuracies across datasets without ranking or calibration.

## Mathematical Foundations & Governing Equations

1. **McNemar's $\chi^2$ Test**: $\chi^2 = \frac{(|b - c| - 1)^2}{b + c}$
2. **Nadeau-Bengio Corrected Variance**: $c = \frac{1}{K} + \frac{n_{\text{test}}}{n_{\text{train}}}, \quad \hat{\sigma} = \sqrt{c \cdot S^2}, \quad t = \frac{\bar{d}}{\hat{\sigma}}$
3. **Friedman Rank Statistic**: $\chi_F^2 = \frac{12 N}{K(K+1)} \left( \sum_{j=1}^K R_j^2 - \frac{K(K+1)^2}{4} \right)$
4. **Bayesian ROPE**: $P(\text{ROPE}) = \Phi\left(\frac{r - \bar{d}}{\hat{\sigma}}\right) - \Phi\left(\frac{-r - \bar{d}}{\hat{\sigma}}\right)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_mcnemar(b, c):
    diff = torch.abs(b - c) - 1.0
    return (diff * diff) / (b + c + 1e-8)

def calc_nadeau_bengio(diffs, n_train=800.0, n_test=200.0):
    k = float(diffs.shape[0])
    d_bar = diffs.mean()
    s_sq = torch.sum((diffs - d_bar)**2) / (k - 1.0)
    c = 1.0 / k + float(n_test) / float(n_train)
    sigma = torch.sqrt(c * s_sq + 1e-8)
    return d_bar / sigma, sigma

def calc_friedman(ranks):
    n, k = float(ranks.shape[0]), float(ranks.shape[1])
    avg_r = ranks.mean(dim=0)
    sum_sq = torch.sum(avg_r * avg_r)
    chi_f = (12.0 * n / (k * (k + 1.0))) * (sum_sq - (k * (k + 1.0)**2) / 4.0)
    return chi_f, avg_r

def calc_bayesian_rope(diffs, n_train=800.0, n_test=200.0, rope_radius=0.01):
    k = float(diffs.shape[0])
    d_bar = diffs.mean()
    s_sq = torch.sum((diffs - d_bar)**2) / (k - 1.0)
    sigma = torch.sqrt((1.0 / k + float(n_test) / float(n_train)) * s_sq + 1e-8)
    def norm_cdf(z): return 0.5 * (1.0 + torch.erf(z / 1.4142135623730951))
    phi1 = norm_cdf((rope_radius - d_bar) / sigma)
    phi2 = norm_cdf((-rope_radius - d_bar) / sigma)
    return torch.stack([1.0 - phi1, phi1 - phi2, phi2])

In [ ]:
# Parity verification against golden reference tensors
mcn = calc_mcnemar(weights["mcnemar_b"], weights["mcnemar_c"])
assert torch.max(torch.abs(mcn - weights["mcnemar_stat_ref"])) < 1e-5, "McNemar mismatch"

nb_t, nb_sig = calc_nadeau_bengio(weights["cv_diffs"], 800.0, 200.0)
assert torch.max(torch.abs(nb_t - weights["nb_t_ref"])) < 1e-5, "NB t mismatch"
assert torch.max(torch.abs(nb_sig - weights["nb_sigma_ref"])) < 1e-5, "NB sigma mismatch"

chi_f, avg_r = calc_friedman(weights["ranks"])
assert torch.max(torch.abs(chi_f - weights["friedman_stat_ref"])) < 1e-5, "Friedman stat mismatch"
assert torch.max(torch.abs(avg_r - weights["avg_ranks_ref"])) < 1e-5, "Avg ranks mismatch"

rope_probs = calc_bayesian_rope(weights["cv_diffs"], 800.0, 200.0, 0.01)
assert torch.max(torch.abs(rope_probs - weights["rope_probs_ref"])) < 1e-5, "ROPE mismatch"
print("All McNemar, Nadeau-Bengio, Friedman, and ROPE assertions passed!")

In [ ]:
# Benchmarking statistical comparison operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_mcnemar(weights["mcnemar_b"], weights["mcnemar_c"])
    _ = calc_nadeau_bengio(weights["cv_diffs"], 800.0, 200.0)
    _ = calc_friedman(weights["ranks"])
    _ = calc_bayesian_rope(weights["cv_diffs"], 800.0, 200.0, 0.01)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused McNemar + Nadeau-Bengio + Friedman + ROPE latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Uncorrected Paired t-Test Inflating Significance on Resampled CV
diffs = weights["cv_diffs"]
k = float(diffs.shape[0])
d_bar = diffs.mean()
s = torch.std(diffs, unbiased=True)

# Uncorrected naive t-test assumes completely independent samples: SE = S / sqrt(K)
se_naive = s / (k**0.5)
t_naive = (d_bar / se_naive).item()

# Nadeau-Bengio corrected SE accounts for overlapping training sets
t_corrected, se_corrected = calc_nadeau_bengio(diffs, n_train=800.0, n_test=200.0)
t_corr_val = t_corrected.item()
se_corr_val = se_corrected.item()

print(f"Naive SE: {se_naive.item():.5f} -> Naive t: {t_naive:.3f}")
print(f"Corrected SE: {se_corr_val:.5f} -> Corrected t: {t_corr_val:.3f}")
assert t_naive > 1.3 * t_corr_val, "Naive t-test severely inflates test statistics and false positive rates!"
assert se_corr_val > se_naive.item(), "Corrected standard error must adjust for shared training data overlap!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **McNemar's Test** conditions on discordant evaluation pairs to eliminate concordant noise.
2. **Nadeau-Bengio Correction** adjusts standard errors for training set overlap in resampled cross-validation.
3. **Friedman Rank Test** provides non-parametric multi-classifier comparisons free from Gaussianity assumptions.
4. **Bayesian ROPE** replaces binary p-values with actionable posterior probabilities over practical equivalence.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.md](../../markdown/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.md)
- **PyTorch 2.x Module:** [11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.py](../../pytorch2/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.py)
- **Functional JAX Module:** [11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.py](../../jax/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.py)
- **Native Rust Candle Track:** [11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.rs](../../candle/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.rs)
- **Hardware Kernel (CPU / WGSL):** [11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.rs](../../kernels/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.rs)
- **Golden SafeTensors:** [11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.safetensors](../../weights/10_xai_and_ai_analysis_methods/11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 10. AI Evaluation Science Bradley Terry IRT Dawid Skene and Min-K%](10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 12. Topological Data Analysis Persistent Homology and Singular Learning LLC](12_topological_data_analysis_persistent_homology_and_singular_learning_llc.ipynb) &rarr;